In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import preprocessing, features

DATA_DIR = '../datasets/rossmann-store-sales'
STORE_FILE = os.path.join(DATA_DIR, 'store.csv')
TRAIN_FILE = os.path.join(DATA_DIR, 'train.csv')
TEST_FILE = os.path.join(DATA_DIR, 'test.csv')
FORECAST_HORIZON = 6*7 # We're forecasting daily for 6 weeks into the future

from pandas.tseries.offsets import DateOffset

def make_targets(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    """
    Generate a target DataFrame containing sales shifted consecutive days ahead up to a
    given forecasting horizon.

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame containing historical sales data.
        Expected to have date `Date` (Datetime), store ID `Store` (str), and `Sales` columns.
    days_ahead : int
        Number of days ahead to shift the sales data (e.g., 1 for next-day sales).

    Returns
    -------
    pd.DataFrame
        A transformed DataFrame containing:
        - 'Store' and 'Date' as index columns.
        - A single target column named `Sales_ahead_{days_ahead}` containing
          the shifted sales values.
    """
        
    def make_target(sales_df: pd.DataFrame, days_ahead: int) -> pd.DataFrame:
        """ Generate a target DataFrame containing sales shifted by a specified number of days ahead. """

        targets = (sales_df
                .shift(freq=DateOffset(days=-days_ahead))
                .iloc[days_ahead:]
                .reset_index()
                .melt(id_vars=['Date'], value_name=f'Sales_ahead_{days_ahead}')
                .set_index(['Store', 'Date'])
                )

        return targets

    # Shift sales various times to gather all targets for the forecasting horizon
    sales_df = pd.pivot(df, index='Date', columns='Store', values='Sales')
    targets = [make_target(sales_df, day) for day in range(1, horizon+1)]

    # Merge and filter Store-Date combinations that do not appear in the training set
    targets = pd.concat(targets, axis=1).reset_index()
    targets = df.merge(targets, on=['Store', 'Date'], how='left').loc[:, targets.columns]

    return targets


In [2]:
store_df = pd.read_csv(STORE_FILE)
store_df = preprocessing.store_data(store_df)

# NOTE: train_df['Open'] == 0 -> train_df['Sales'] = 0. This happens always
train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1) # Not available in test
train_df = features.attach_store_data(train_df, store_df)

test_df = pd.read_csv(TEST_FILE, index_col=0, parse_dates=['Date'])
test_df = features.attach_store_data(test_df, store_df)

C:\Users\m_kal\AppData\Local\Temp\ipykernel_24396\2952918199.py:5: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_FILE, parse_dates=['Date']).drop(['Customers'], axis=1) # Not available in test


In [3]:
""" Feature engineering """

# Competition-related features
train_df['CompetitionDistance'] = train_df['CompetitionDistance'].apply(np.log1p)
train_df['CompetitionSinceMonths'] = ( (train_df['Date'] - train_df['CompetitionSinceDate']).dt.days / 30.0 ).round()

# Promotion related features
train_df['Promo2SinceWeeks'] =  ( (train_df['Date'] - train_df['Promo2SinceDate']).dt.days / 7.0 ).fillna(0).round().astype(int) * train_df['Promo2']

# Basic date features
train_df['WeekOfYear'] = train_df['Date'].dt.isocalendar().week
train_df['Month'] = train_df['Date'].dt.month
train_df['Year'] = train_df['Date'].dt.year
train_df['Quarter'] = train_df['Date'].dt.quarter

# Calendar and seasonality features
train_df['is_weekend'] = train_df['Date'].dt.dayofweek >= 5

# Cyclical features
train_df['Month_sin'] = np.sin(2 * np.pi * train_df['Month'] / 12)
train_df['Month_cos'] = np.cos(2 * np.pi * train_df['Month'] / 12)
train_df['Dayofweek_sin'] = np.sin(2 * np.pi * train_df['DayOfWeek'] / 7)
train_df['Dayofweek_cos'] = np.cos(2 * np.pi * train_df['DayOfWeek'] / 7)

# Drop useless
#train_df.drop(['Promo2SinceDate', 'CompetitionSinceDate'], axis=1, inplace=True)


In [4]:

# Generate target dataframe
targets = make_targets(train_df[['Store', 'Date', 'Sales']], horizon=FORECAST_HORIZON)

In [ ]:

# TODO: Lag and rolling features - NOTE: We are predicting 6 weeks ahead.
train_df['lag_1'] = train_df['target'].shift(1)
train_df['rolling_mean_7'] = train_df['target'].shift(1).rolling(7).mean()
train_df['rolling_std_7'] = train_df['target'].shift(1).rolling(7).std()
